# 🚌 Klagenfurt FleetPy Simulation Analysis

## Professional Fleet Simulation Analysis using FleetPy Framework

This notebook analyzes the results from FleetPy simulations of different fleet sizes in Klagenfurt. FleetPy is a professional fleet simulation framework that provides realistic vehicle routing, demand-supply matching, and operational constraints.

**FleetPy Simulation Scenarios:**
- **Scenario 1**: 20 vehicles fleet
- **Scenario 2**: 30 vehicles fleet  
- **Scenario 3**: 40 vehicles fleet

**Analysis Focus:**
- 📊 Service quality and fleet utilization across scenarios
- ⏱️ Temporal patterns and peak demand analysis
- 🚗 Vehicle performance and operational metrics
- 💰 Cost-effectiveness and recommendations

> **Note**: This analysis uses real FleetPy simulation results, not synthetic data. Run `python run_klagenfurt.py` to generate fresh simulation results if needed.

In [1]:
# Import required libraries for FleetPy results analysis
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from IPython.display import display

# Set up plotting style for professional visualizations
plt.style.use('default')
plt.rcParams['figure.figsize'] = [15, 10]
plt.rcParams['lines.linewidth'] = 2
plt.rcParams['lines.markersize'] = 8
plt.rcParams['axes.grid'] = True
sns.set_style("whitegrid")

# Set up directories
results_dir = os.path.join(os.path.dirname(os.getcwd()), "results")
analysis_results_dir = os.path.join(results_dir, "analysis")
os.makedirs(analysis_results_dir, exist_ok=True)

print(f"📁 FleetPy results directory: {results_dir}")
print(f"📁 Analysis outputs will be saved to: {analysis_results_dir}")
print("✅ Libraries imported and environment configured")

📁 FleetPy results directory: /Users/danieldeutsch/Desktop/code_projects/FleetPy/studies/klagenfurt_case_study/results
📁 Analysis outputs will be saved to: /Users/danieldeutsch/Desktop/code_projects/FleetPy/studies/klagenfurt_case_study/results/analysis
✅ Libraries imported and environment configured


## 1. FleetPy Results Loading and Processing

Load and process the FleetPy simulation results from different scenarios. Each scenario represents a different fleet size configuration:

- **🚌 Fleet Sizes**: 20, 30, 40 vehicles
- **📊 Data Sources**: User statistics, vehicle performance, operational metrics
- **🕐 Time Analysis**: Peak demand patterns and service quality over time

In [2]:
# 📊 FLEETPY RESULTS LOADER (IMPROVED)
print("="*80)
print("🚌 KLAGENFURT FLEETPY SIMULATION ANALYSIS")
print("="*80)

class FleetPyResultsAnalyzer:
    """
    Analyzes FleetPy simulation results from multiple scenarios
    Improved version with better error handling and debugging
    """

    def __init__(self, results_base_dir):
        self.results_base_dir = results_base_dir
        self.scenarios = {
            'klagenfurt_scenario_1': {'fleet_size': 20, 'name': 'Small Fleet (20 vehicles)'},
            'klagenfurt_scenario_2': {'fleet_size': 30, 'name': 'Medium Fleet (30 vehicles)'},
            'klagenfurt_scenario_3': {'fleet_size': 40, 'name': 'Large Fleet (40 vehicles)'}
        }
        self.results = {}

    def load_scenario_data(self, scenario_name):
        """Load all relevant data for a specific scenario with improved error handling"""
        scenario_dir = os.path.join(self.results_base_dir, scenario_name)
        print(f"    📂 Checking directory: {scenario_dir}")

        if not os.path.exists(scenario_dir):
            print(f"    ⚠️  Directory not found: {scenario_dir}")
            return None

        data = {}

        # Load user statistics (trip requests and outcomes)
        user_stats_file = os.path.join(scenario_dir, "1_user-stats.csv")
        print(f"    📄 Loading user stats from: {user_stats_file}")

        if os.path.exists(user_stats_file):
            try:
                data['user_stats'] = pd.read_csv(user_stats_file)
                print(f"    ✅ Loaded {len(data['user_stats'])} user requests")

                # Quick preview of modal states
                if 'modal_state' in data['user_stats'].columns:
                    modal_counts = data['user_stats']['modal_state'].value_counts()
                    print(f"    📊 Modal states: {dict(modal_counts)}")

            except Exception as e:
                print(f"    ❌ Error loading user stats: {e}")
                return None
        else:
            print(f"    ⚠️  User stats file not found")

        # Load vehicle evaluation data
        veh_eval_file = os.path.join(scenario_dir, "standard_mod-0.0_veh_eval.csv")
        print(f"    🚗 Loading vehicle data from: {veh_eval_file}")

        if os.path.exists(veh_eval_file):
            try:
                data['vehicle_eval'] = pd.read_csv(veh_eval_file)
                print(f"    ✅ Loaded {len(data['vehicle_eval'])} vehicles")

                # Quick preview of vehicle usage
                if 'total km' in data['vehicle_eval'].columns:
                    active_vehs = len(data['vehicle_eval'][data['vehicle_eval']['total km'] > 0])
                    print(f"    🚙 Active vehicles: {active_vehs}/{len(data['vehicle_eval'])}")

            except Exception as e:
                print(f"    ❌ Error loading vehicle data: {e}")
        else:
            print(f"    ⚠️  Vehicle eval file not found")

        # Load standard evaluation summary
        std_eval_file = os.path.join(scenario_dir, "standard_eval.csv")
        if os.path.exists(std_eval_file):
            try:
                data['standard_eval'] = pd.read_csv(std_eval_file, index_col=0)
                print(f"    ✅ Loaded evaluation summary")
            except Exception as e:
                print(f"    ❌ Error loading standard eval: {e}")

        # Load configuration
        config_file = os.path.join(scenario_dir, "00_config.json")
        if os.path.exists(config_file):
            try:
                with open(config_file, 'r') as f:
                    data['config'] = json.load(f)
                print(f"    ✅ Loaded configuration")
            except Exception as e:
                print(f"    ❌ Error loading config: {e}")

        return data

    def analyze_scenario_performance(self, scenario_name, data):
        """Analyze performance metrics for a scenario with better error handling"""
        print(f"    🔍 Analyzing performance for {scenario_name}")

        if data is None or 'user_stats' not in data:
            print(f"    ❌ No user stats data available for analysis")
            return None

        user_stats = data['user_stats']
        fleet_size = self.scenarios[scenario_name]['fleet_size']

        try:
            # Basic service metrics
            total_requests = len(user_stats)
            print(f"    📊 Total requests: {total_requests}")

            # Check if modal_state column exists
            if 'modal_state' not in user_stats.columns:
                print(f"    ⚠️  Warning: 'modal_state' column not found in user stats")
                served_requests = 0
                service_rate = 0
            else:
                served_requests = len(user_stats[user_stats['modal_state'] == 1])
                service_rate = (served_requests / total_requests * 100) if total_requests > 0 else 0
                print(f"    ✅ Served requests: {served_requests} ({service_rate:.1f}%)")

            # Waiting time analysis (for served requests only)
            served_data = user_stats[user_stats['modal_state'] == 1] if 'modal_state' in user_stats.columns else pd.DataFrame()

            if len(served_data) > 0 and 'pickup_time' in served_data.columns and 'earliest_pickup_time' in served_data.columns:
                served_data = served_data.copy()
                served_data['waiting_time'] = served_data['pickup_time'] - served_data['earliest_pickup_time']
                # Handle NaN values
                served_data = served_data.dropna(subset=['waiting_time'])

                if len(served_data) > 0:
                    avg_waiting_time = served_data['waiting_time'].mean() / 60  # Convert to minutes
                    max_waiting_time = served_data['waiting_time'].max() / 60
                    quick_service = len(served_data[served_data['waiting_time'] <= 600])
                    quick_service_rate = (quick_service / len(served_data) * 100)
                    print(f"    ⏱️  Average wait: {avg_waiting_time:.1f}min, Max: {max_waiting_time:.1f}min")
                else:
                    avg_waiting_time = float('inf')
                    max_waiting_time = float('inf')
                    quick_service_rate = 0
            else:
                avg_waiting_time = float('inf')
                max_waiting_time = float('inf')
                quick_service_rate = 0
                print(f"    ⚠️  No valid waiting time data available")

            # Temporal analysis
            if 'rq_time' in user_stats.columns:
                user_stats['hour'] = (user_stats['rq_time'] / 3600).astype(int)
                hourly_demand = user_stats['hour'].value_counts().sort_index()

                # Peak time analysis
                morning_peak = user_stats[(user_stats['hour'] >= 7) & (user_stats['hour'] <= 9)]
                afternoon_peak = user_stats[(user_stats['hour'] >= 15) & (user_stats['hour'] <= 18)]

                if 'modal_state' in user_stats.columns:
                    morning_served = len(morning_peak[morning_peak['modal_state'] == 1])
                    afternoon_served = len(afternoon_peak[afternoon_peak['modal_state'] == 1])
                else:
                    morning_served = 0
                    afternoon_served = 0

                morning_service_rate = (morning_served / len(morning_peak) * 100) if len(morning_peak) > 0 else 0
                afternoon_service_rate = (afternoon_served / len(afternoon_peak) * 100) if len(afternoon_peak) > 0 else 0

                print(f"    🌅 Morning peak: {len(morning_peak)} requests, {morning_service_rate:.1f}% served")
                print(f"    🌆 Afternoon peak: {len(afternoon_peak)} requests, {afternoon_service_rate:.1f}% served")
            else:
                hourly_demand = pd.Series()
                morning_peak = pd.DataFrame()
                afternoon_peak = pd.DataFrame()
                morning_service_rate = 0
                afternoon_service_rate = 0

            # Vehicle utilization
            vehicle_data = data.get('vehicle_eval', pd.DataFrame())
            if not vehicle_data.empty and 'total km' in vehicle_data.columns:
                active_vehicles = len(vehicle_data[vehicle_data['total km'] > 0])
                total_km = vehicle_data['total km'].sum()
                avg_km_per_vehicle = total_km / fleet_size
                utilization_rate = (active_vehicles / fleet_size * 100)
                print(f"    🚗 Fleet utilization: {active_vehicles}/{fleet_size} vehicles ({utilization_rate:.1f}%)")
            else:
                active_vehicles = 0
                total_km = 0
                avg_km_per_vehicle = 0
                utilization_rate = 0
                print(f"    ⚠️  No vehicle utilization data available")

            analysis_result = {
                'scenario_name': scenario_name,
                'fleet_size': fleet_size,
                'total_requests': total_requests,
                'served_requests': served_requests,
                'rejected_requests': total_requests - served_requests,
                'service_rate': service_rate,
                'avg_waiting_time': avg_waiting_time,
                'max_waiting_time': max_waiting_time,
                'quick_service_rate': quick_service_rate,
                'morning_peak_requests': len(morning_peak),
                'morning_peak_served': morning_served,
                'morning_service_rate': morning_service_rate,
                'afternoon_peak_requests': len(afternoon_peak),
                'afternoon_peak_served': afternoon_served,
                'afternoon_service_rate': afternoon_service_rate,
                'active_vehicles': active_vehicles,
                'total_km': total_km,
                'avg_km_per_vehicle': avg_km_per_vehicle,
                'utilization_rate': utilization_rate,
                'hourly_demand': hourly_demand,
                'user_stats': user_stats,
                'vehicle_data': vehicle_data
            }

            print(f"    ✅ Analysis completed successfully")
            return analysis_result

        except Exception as e:
            print(f"    ❌ Error during analysis: {e}")
            import traceback
            traceback.print_exc()
            return None

    def load_all_scenarios(self):
        """Load and analyze all scenarios with improved progress tracking"""
        print(f"\n🚀 LOADING FLEETPY SIMULATION RESULTS")
        print(f"📁 Base directory: {self.results_base_dir}")
        print("-" * 60)

        for i, (scenario_name, scenario_info) in enumerate(self.scenarios.items(), 1):
            print(f"\n📊 Processing {i}/3: {scenario_name}")
            print(f"    🏷️  {scenario_info['name']}")

            try:
                # Load data
                data = self.load_scenario_data(scenario_name)

                # Analyze performance
                analysis = self.analyze_scenario_performance(scenario_name, data)

                if analysis:
                    self.results[scenario_name] = analysis
                    print(f"    ✅ SUMMARY: {analysis['served_requests']}/{analysis['total_requests']} requests served ({analysis['service_rate']:.1f}%)")
                else:
                    print(f"    ❌ Analysis failed for {scenario_name}")

            except Exception as e:
                print(f"    ❌ Critical error processing {scenario_name}: {e}")
                import traceback
                traceback.print_exc()

        print("-" * 60)
        print(f"✅ Processing complete! Loaded {len(self.results)}/3 scenarios")

        return self.results

# Initialize analyzer and load results
print("🚀 INITIALIZING FLEETPY RESULTS ANALYZER...")
analyzer = FleetPyResultsAnalyzer(results_dir)
fleetpy_results = analyzer.load_all_scenarios()

# Create summary DataFrame only if we have results
if fleetpy_results:
    print(f"\n📊 CREATING SUMMARY DATAFRAME...")
    summary_data = []

    for scenario_name, result in fleetpy_results.items():
        summary_data.append({
            'Scenario': result['scenario_name'].replace('klagenfurt_scenario_', 'Scenario '),
            'Fleet Size': result['fleet_size'],
            'Total Requests': result['total_requests'],
            'Served Requests': result['served_requests'],
            'Service Rate (%)': round(result['service_rate'], 1),
            'Avg Wait Time (min)': round(result['avg_waiting_time'], 1) if result['avg_waiting_time'] != float('inf') else 'N/A',
            'Active Vehicles': result['active_vehicles'],
            'Fleet Utilization (%)': round(result['utilization_rate'], 1),
            'Morning Peak Service (%)': round(result['morning_service_rate'], 1),
            'Afternoon Peak Service (%)': round(result['afternoon_service_rate'], 1)
        })

    summary_df = pd.DataFrame(summary_data)

    print(f"\n📊 FLEETPY SIMULATION SUMMARY:")
    print("=" * 80)
    display(summary_df)

    # Save summary
    summary_df.to_csv(os.path.join(analysis_results_dir, "fleetpy_analysis_summary.csv"), index=False)
    print(f"\n💾 Summary saved to: {os.path.join(analysis_results_dir, 'fleetpy_analysis_summary.csv')}")
else:
    print("❌ No results to analyze")
    print("💡 Possible issues:")
    print("   • FleetPy simulation didn't run successfully")
    print("   • Results files are corrupted or missing")
    print("   • Wrong file paths or directory structure")

print("=" * 80)

🚌 KLAGENFURT FLEETPY SIMULATION ANALYSIS
🚀 INITIALIZING FLEETPY RESULTS ANALYZER...

🚀 LOADING FLEETPY SIMULATION RESULTS
📁 Base directory: /Users/danieldeutsch/Desktop/code_projects/FleetPy/studies/klagenfurt_case_study/results
------------------------------------------------------------

📊 Processing 1/3: klagenfurt_scenario_1
    🏷️  Small Fleet (20 vehicles)
    📂 Checking directory: /Users/danieldeutsch/Desktop/code_projects/FleetPy/studies/klagenfurt_case_study/results/klagenfurt_scenario_1
    📄 Loading user stats from: /Users/danieldeutsch/Desktop/code_projects/FleetPy/studies/klagenfurt_case_study/results/klagenfurt_scenario_1/1_user-stats.csv
    ✅ Loaded 100 user requests
    📊 Modal states: {0: np.int64(100)}
    🚗 Loading vehicle data from: /Users/danieldeutsch/Desktop/code_projects/FleetPy/studies/klagenfurt_case_study/results/klagenfurt_scenario_1/standard_mod-0.0_veh_eval.csv
    ✅ Loaded 20 vehicles
    🚙 Active vehicles: 1/20
    ✅ Loaded evaluation summary
    ✅ Load

,Scenario,Fleet Size,Total Requests,Served Requests,Service Rate (%),Avg Wait Time (min),Active Vehicles,Fleet Utilization (%),Morning Peak Service (%),Afternoon Peak Service (%)
0,Scenario 1,20,100,0,0.0,N/A,1,5.0,0.0,0
1,Scenario 2,30,100,0,0.0,N/A,2,6.7,0.0,0
2,Scenario 3,40,100,0,0.0,N/A,2,5.0,0.0,0



💾 Summary saved to: /Users/danieldeutsch/Desktop/code_projects/FleetPy/studies/klagenfurt_case_study/results/analysis/fleetpy_analysis_summary.csv


In [3]:
# 🔍 DETAILED FLEETPY ANALYSIS (FIXED FORMATTING)
print("="*80)
print("🔍 DETAILED FLEETPY PERFORMANCE ANALYSIS")
print("="*80)

def detailed_temporal_analysis(fleetpy_results):
    """Perform detailed temporal analysis of FleetPy results with fixed formatting"""

    print("\n📈 TEMPORAL DEMAND AND SERVICE PATTERNS:")
    print("-" * 50)

    # Collect temporal data from all scenarios
    temporal_data = {}

    for scenario_name, result in fleetpy_results.items():
        print(f"   🔄 Processing {result['fleet_size']} vehicles scenario...")

        if 'user_stats' in result and not result['user_stats'].empty:
            user_stats = result['user_stats'].copy()  # Work with a copy
            fleet_size = result['fleet_size']

            # Ensure hour column exists
            if 'hour' not in user_stats.columns:
                if 'rq_time' in user_stats.columns:
                    print(f"   📊 Creating hour column from rq_time...")
                    user_stats['hour'] = (user_stats['rq_time'] / 3600).astype(int)
                else:
                    print(f"   ⚠️  No time data available for {fleet_size} vehicles scenario")
                    continue

            # Check for required columns
            required_cols = ['modal_state', 'hour']
            missing_cols = [col for col in required_cols if col not in user_stats.columns]
            if missing_cols:
                print(f"   ⚠️  Missing columns {missing_cols} in {fleet_size} vehicles scenario")
                continue

            print(f"   📊 Analyzing hourly patterns for {len(user_stats)} requests...")

            # Optimized hourly analysis using vectorized operations
            hourly_stats = []
            hours_with_data = user_stats['hour'].unique()

            for hour in range(6, 23):  # 6 AM to 10 PM
                if hour in hours_with_data:
                    hour_mask = user_stats['hour'] == hour
                    hour_data = user_stats[hour_mask]

                    hour_requests = len(hour_data)
                    hour_served = (hour_data['modal_state'] == 1).sum()
                    hour_service_rate = (hour_served / hour_requests * 100) if hour_requests > 0 else 0

                    # Calculate average waiting time for this hour (optimized)
                    avg_wait = 0
                    if hour_served > 0 and 'pickup_time' in hour_data.columns and 'earliest_pickup_time' in hour_data.columns:
                        served_mask = hour_data['modal_state'] == 1
                        served_data = hour_data[served_mask]

                        # Vectorized waiting time calculation
                        waiting_times = served_data['pickup_time'] - served_data['earliest_pickup_time']
                        waiting_times = waiting_times.dropna()

                        if len(waiting_times) > 0:
                            avg_wait = waiting_times.mean() / 60  # Convert to minutes

                    hourly_stats.append({
                        'hour': hour,  # Store as integer
                        'requests': hour_requests,
                        'served': hour_served,
                        'service_rate': hour_service_rate,
                        'avg_waiting_time': avg_wait
                    })
                else:
                    # No data for this hour
                    hourly_stats.append({
                        'hour': hour,  # Store as integer
                        'requests': 0,
                        'served': 0,
                        'service_rate': 0,
                        'avg_waiting_time': 0
                    })

            temporal_data[f"{fleet_size}_vehicles"] = pd.DataFrame(hourly_stats)

            # Print summary for this scenario
            print(f"\n🚌 {fleet_size} Vehicles Fleet Analysis:")

            # Peak hours identification (limit to top 3 to avoid excessive output)
            hourly_df = pd.DataFrame(hourly_stats)
            peak_hours = hourly_df.nlargest(3, 'requests')

            print(f"   📊 Top 3 demand hours:")
            for _, peak in peak_hours.iterrows():
                if peak['requests'] > 0:
                    # Fixed: Convert to int explicitly before formatting
                    hour_int = int(peak['hour'])
                    requests_int = int(peak['requests'])
                    print(f"      {hour_int:02d}:00 - {requests_int} requests ({peak['service_rate']:.1f}% served)")

            # Service quality during peaks (vectorized)
            morning_peak_mask = (hourly_df['hour'] >= 7) & (hourly_df['hour'] <= 9)
            afternoon_peak_mask = (hourly_df['hour'] >= 15) & (hourly_df['hour'] <= 18)

            morning_peak_data = hourly_df[morning_peak_mask]
            afternoon_peak_data = hourly_df[afternoon_peak_mask]

            if len(morning_peak_data) > 0:
                morning_avg_service = morning_peak_data['service_rate'].mean()
                print(f"   🌅 Morning peak (7-9h): {morning_avg_service:.1f}% average service rate")

            if len(afternoon_peak_data) > 0:
                afternoon_avg_service = afternoon_peak_data['service_rate'].mean()
                print(f"   🌆 Afternoon peak (15-18h): {afternoon_avg_service:.1f}% average service rate")

        else:
            print(f"   ⚠️  No user stats available for {result.get('fleet_size', 'unknown')} vehicles scenario")

    print(f"\n✅ Temporal analysis completed for {len(temporal_data)} scenarios")
    return temporal_data

def fleet_utilization_analysis(fleetpy_results):
    """Analyze fleet utilization patterns with optimized performance"""

    print(f"\n🚗 FLEET UTILIZATION ANALYSIS:")
    print("-" * 50)

    utilization_data = []

    for scenario_name, result in fleetpy_results.items():
        fleet_size = result['fleet_size']
        print(f"   🔄 Analyzing {fleet_size} vehicles utilization...")

        print(f"\n🚌 {fleet_size} Vehicles Fleet:")
        print(f"   🔧 Active vehicles: {result['active_vehicles']}/{fleet_size} ({result['utilization_rate']:.1f}%)")
        print(f"   📏 Total distance: {result['total_km']:.1f} km")
        print(f"   📊 Average km per vehicle: {result['avg_km_per_vehicle']:.1f} km")

        # Calculate efficiency metrics
        served_requests = result['served_requests']
        total_km = result['total_km']

        if served_requests > 0 and total_km > 0:
            km_per_trip = total_km / served_requests
            requests_per_active_vehicle = served_requests / result['active_vehicles'] if result['active_vehicles'] > 0 else 0

            print(f"   🎯 Efficiency: {km_per_trip:.1f} km per served trip")
            print(f"   📈 Productivity: {requests_per_active_vehicle:.1f} trips per active vehicle")

        utilization_data.append({
            'fleet_size': fleet_size,
            'active_vehicles': result['active_vehicles'],
            'utilization_rate': result['utilization_rate'],
            'total_km': result['total_km'],
            'served_requests': served_requests
        })

    return pd.DataFrame(utilization_data)

def service_quality_comparison(fleetpy_results):
    """Compare service quality across fleet sizes with optimized processing"""

    print(f"\n🏆 SERVICE QUALITY COMPARISON:")
    print("-" * 50)

    if not fleetpy_results:
        print("   ⚠️  No results to compare")
        return

    # Find best performers
    results_list = list(fleetpy_results.values())
    best_service_rate = max(results_list, key=lambda x: x['service_rate'])
    best_utilization = max(results_list, key=lambda x: x['utilization_rate'])

    # Find most efficient (best service with fewest vehicles)
    valid_results = [r for r in results_list if r['avg_waiting_time'] != float('inf')]
    if valid_results:
        best_efficiency = min(valid_results, key=lambda x: x['fleet_size'] / max(x['service_rate'], 1))

        print(f"🏆 PERFORMANCE CHAMPIONS:")
        print(f"   🥇 Best Service Rate: {best_service_rate['fleet_size']} vehicles ({best_service_rate['service_rate']:.1f}%)")
        print(f"   🚗 Best Fleet Utilization: {best_utilization['fleet_size']} vehicles ({best_utilization['utilization_rate']:.1f}%)")
        print(f"   ⚡ Most Efficient: {best_efficiency['fleet_size']} vehicles (best service-to-fleet ratio)")

    # Service quality assessment
    print(f"\n📊 DETAILED SERVICE ASSESSMENT:")
    for scenario_name, result in sorted(fleetpy_results.items()):
        fleet_size = result['fleet_size']
        service_rate = result['service_rate']

        # Quality rating
        if service_rate >= 90:
            quality = "🌟 Excellent"
        elif service_rate >= 75:
            quality = "✅ Good"
        elif service_rate >= 50:
            quality = "⚠️ Fair"
        elif service_rate >= 25:
            quality = "❌ Poor"
        else:
            quality = "🚫 Critical"

        print(f"   {fleet_size} vehicles: {service_rate:.1f}% service rate - {quality}")

        # Specific recommendations
        if result['utilization_rate'] < 50 and service_rate > 80:
            print(f"      💡 Over-sized fleet - consider reducing to improve efficiency")
        elif result['utilization_rate'] > 80 and service_rate < 60:
            print(f"      💡 Under-sized fleet - consider increasing for better service")
        elif 50 <= result['utilization_rate'] <= 80 and service_rate >= 70:
            print(f"      ✅ Well-balanced fleet size")

# Perform detailed analysis with error handling
print("🚀 Starting detailed analysis...")

try:
    if fleetpy_results:
        print(f"📊 Analyzing {len(fleetpy_results)} scenarios...")

        # Run analyses with progress tracking
        temporal_data = detailed_temporal_analysis(fleetpy_results)
        utilization_df = fleet_utilization_analysis(fleetpy_results)
        service_quality_comparison(fleetpy_results)

        # Save detailed analysis
        if not utilization_df.empty:
            utilization_df.to_csv(os.path.join(analysis_results_dir, "fleet_utilization_analysis.csv"), index=False)
            print(f"\n💾 Detailed analysis saved to: {analysis_results_dir}")

        print(f"\n✅ DETAILED ANALYSIS COMPLETED SUCCESSFULLY")

    else:
        print("❌ No results available for detailed analysis")
        print("💡 Check if the previous FleetPy analysis cell ran successfully")

except Exception as e:
    print(f"❌ Error during detailed analysis: {e}")
    print(f"💡 This might be due to:")
    print(f"   • Missing or corrupted data files")
    print(f"   • Memory issues with large datasets")
    print(f"   • Column naming inconsistencies")

    # Debug information
    if 'fleetpy_results' in locals():
        print(f"\n🔍 Debug info:")
        print(f"   • Number of scenarios: {len(fleetpy_results)}")
        for name, result in fleetpy_results.items():
            if 'user_stats' in result:
                print(f"   • {name}: {len(result['user_stats'])} records")
                print(f"     Columns: {list(result['user_stats'].columns)[:5]}...")

    import traceback
    traceback.print_exc()

print("=" * 80)

🔍 DETAILED FLEETPY PERFORMANCE ANALYSIS
🚀 Starting detailed analysis...
📊 Analyzing 3 scenarios...

📈 TEMPORAL DEMAND AND SERVICE PATTERNS:
--------------------------------------------------
   🔄 Processing 20 vehicles scenario...
   📊 Analyzing hourly patterns for 100 requests...

🚌 20 Vehicles Fleet Analysis:
   📊 Top 3 demand hours:
      08:00 - 56 requests (0.0% served)
      07:00 - 44 requests (0.0% served)
   🌅 Morning peak (7-9h): 0.0% average service rate
   🌆 Afternoon peak (15-18h): 0.0% average service rate
   🔄 Processing 30 vehicles scenario...
   📊 Analyzing hourly patterns for 100 requests...

🚌 30 Vehicles Fleet Analysis:
   📊 Top 3 demand hours:
      08:00 - 56 requests (0.0% served)
      07:00 - 44 requests (0.0% served)
   🌅 Morning peak (7-9h): 0.0% average service rate
   🌆 Afternoon peak (15-18h): 0.0% average service rate
   🔄 Processing 40 vehicles scenario...
   📊 Analyzing hourly patterns for 100 requests...

🚌 40 Vehicles Fleet Analysis:
   📊 Top 3 demand

In [ ]:
# ? DIAGNOSTIC ANALYSIS: Why 0% Service Rate?
print("="*80)
print("? DIAGNOSTIC ANALYSIS: INVESTIGATING 0% SERVICE RATE ISSUE")
print("="*80)

if fleetpy_results:
    print(f"📊 Found {len(fleetpy_results)} scenarios to investigate...")

    for scenario_name, result in fleetpy_results.items():
        fleet_size = result['fleet_size']
        print(f"\n🚌 SCENARIO: {scenario_name} ({fleet_size} vehicles)")
        print("-" * 50)

        # Basic stats
        print(f"📊 Basic Stats:")
        print(f"   • Total requests: {result['total_requests']}")
        print(f"   • Served requests: {result['served_requests']}")
        print(f"   • Service rate: {result['service_rate']:.1f}%")
        print(f"   • Active vehicles: {result['active_vehicles']}")

        # Examine user_stats data
        if 'user_stats' in result and not result['user_stats'].empty:
            user_stats = result['user_stats']
            print(f"\n📄 User Stats Data Analysis:")
            print(f"   • DataFrame shape: {user_stats.shape}")
            print(f"   • Columns: {list(user_stats.columns)}")

            # Check modal_state values
            if 'modal_state' in user_stats.columns:
                modal_counts = user_stats['modal_state'].value_counts().sort_index()
                print(f"   • Modal state distribution: {dict(modal_counts)}")

                # Show first few records to understand data
                print(f"\n📋 Sample Records (first 5):")
                sample_cols = ['request_id', 'modal_state', 'rq_time', 'earliest_pickup_time']
                available_cols = [col for col in sample_cols if col in user_stats.columns]
                if available_cols:
                    print(user_stats[available_cols].head().to_string())

                # Check if there are any served requests (modal_state = 1)
                served_requests = user_stats[user_stats['modal_state'] == 1]
                if len(served_requests) > 0:
                    print(f"\n✅ Found {len(served_requests)} served requests!")
                    print("📋 Sample served request:")
                    print(served_requests[available_cols].iloc[0].to_string())
                else:
                    print(f"\n❌ NO SERVED REQUESTS FOUND!")
                    print("🔍 Checking modal_state values:")
                    unique_states = user_stats['modal_state'].unique()
                    print(f"   • Unique modal_state values: {sorted(unique_states)}")

                    # Check if modal_state might be coded differently
                    print(f"\n🔍 Value Counts for modal_state:")
                    for state, count in modal_counts.items():
                        print(f"      modal_state = {state}: {count} records")

            else:
                print(f"   ❌ ERROR: 'modal_state' column missing!")
                print(f"   Available columns: {list(user_stats.columns)}")

        # Examine vehicle data
        if 'vehicle_data' in result and not result['vehicle_data'].empty:
            vehicle_data = result['vehicle_data']
            print(f"\n🚗 Vehicle Data Analysis:")
            print(f"   • Vehicle DataFrame shape: {vehicle_data.shape}")
            print(f"   • Columns: {list(vehicle_data.columns)}")

            if 'total km' in vehicle_data.columns:
                total_km_stats = vehicle_data['total km'].describe()
                print(f"   • Total km statistics:")
                print(f"     - Mean: {total_km_stats['mean']:.2f} km")
                print(f"     - Max: {total_km_stats['max']:.2f} km")
                print(f"     - Vehicles with >0 km: {len(vehicle_data[vehicle_data['total km'] > 0])}")
        else:
            print(f"\n⚠️  No vehicle data available")

    print(f"\n🔍 ROOT CAUSE ANALYSIS:")
    print("="*50)

    # Analyze patterns across scenarios
    all_served = sum(result['served_requests'] for result in fleetpy_results.values())
    all_total = sum(result['total_requests'] for result in fleetpy_results.values())

    if all_served == 0:
        print("❌ CRITICAL ISSUE: NO REQUESTS SERVED IN ANY SCENARIO")
        print("\n💡 POSSIBLE CAUSES:")
        print("1. 🚫 Modal state interpretation error:")
        print("   - modal_state = 1 might not mean 'served'")
        print("   - Check FleetPy documentation for correct modal_state values")

        print("\n2. ⏰ Timing/coordination issues:")
        print("   - Vehicles not initialized properly")
        print("   - Demand and supply timing mismatch")

        print("\n3. 🗺️ Geographic/network issues:")
        print("   - Vehicles and requests in different locations")
        print("   - Network routing not working")

        print("\n4. 📊 Data processing error:")
        print("   - Wrong column interpretation in analysis")
        print("   - FleetPy results format changed")

        # Check if modal_state values are consistent
        modal_values_by_scenario = {}
        for scenario_name, result in fleetpy_results.items():
            if 'user_stats' in result and 'modal_state' in result['user_stats'].columns:
                modal_values_by_scenario[scenario_name] = result['user_stats']['modal_state'].unique()

        print(f"\n🔍 MODAL STATE INVESTIGATION:")
        for scenario, values in modal_values_by_scenario.items():
            print(f"   {scenario}: modal_state values = {sorted(values)}")

        # Suggest immediate fix
        print(f"\n🛠️  IMMEDIATE DIAGNOSTIC STEPS:")
        print("1. Check FleetPy modal_state documentation")
        print("2. Try different modal_state values (0, 2, 3, etc.)")
        print("3. Examine actual FleetPy output files")
        print("4. Verify simulation completed successfully")

    else:
        print(f"✅ Found {all_served}/{all_total} total served requests across scenarios")

    # Quick visualization fix - create simple diagnostic plots
    print(f"\n📊 CREATING SIMPLE DIAGNOSTIC VISUALIZATION...")

    try:
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))

        # Plot 1: Requests by scenario
        scenarios = list(fleetpy_results.keys())
        total_requests = [fleetpy_results[s]['total_requests'] for s in scenarios]
        served_requests = [fleetpy_results[s]['served_requests'] for s in scenarios]

        ax1 = axes[0]
        x = range(len(scenarios))
        ax1.bar(x, total_requests, label='Total Requests', alpha=0.7)
        ax1.bar(x, served_requests, label='Served Requests', alpha=0.9)
        ax1.set_title('Requests by Scenario')
        ax1.set_xticks(x)
        ax1.set_xticklabels([f"Scenario {i+1}" for i in x])
        ax1.legend()
        ax1.grid(True, alpha=0.3)

        # Plot 2: Fleet utilization
        fleet_sizes = [fleetpy_results[s]['fleet_size'] for s in scenarios]
        active_vehicles = [fleetpy_results[s]['active_vehicles'] for s in scenarios]

        ax2 = axes[1]
        ax2.bar(x, fleet_sizes, label='Total Fleet', alpha=0.7)
        ax2.bar(x, active_vehicles, label='Active Vehicles', alpha=0.9)
        ax2.set_title('Fleet Utilization')
        ax2.set_xticks(x)
        ax2.set_xticklabels([f"{fleet_sizes[i]}v" for i in x])
        ax2.legend()
        ax2.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()

        print("✅ Diagnostic visualization created successfully!")

    except Exception as e:
        print(f"❌ Visualization error: {e}")
        print("💡 This is expected if there are data issues")

else:
    print("❌ No FleetPy results available for diagnosis")

print("="*80)

In [ ]:
# 🎯 FLEETPY SIMULATION RECOMMENDATIONS

if fleetpy_results:
    print("="*80)
    print("🎯 KLAGENFURT FLEETPY SIMULATION RECOMMENDATIONS")
    print("="*80)

    # Find optimal configurations
    valid_results = [r for r in fleetpy_results.values() if r['service_rate'] > 0]

    if valid_results:
        best_service = max(valid_results, key=lambda x: x['service_rate'])
        best_utilization = max(valid_results, key=lambda x: x['utilization_rate'])

        # Calculate efficiency score (service rate per vehicle)
        for result in valid_results:
            result['efficiency_score'] = result['service_rate'] / result['fleet_size']

        most_efficient = max(valid_results, key=lambda x: x['efficiency_score'])

        print(f"\n🏆 PERFORMANCE ANALYSIS:")
        print(f"   🥇 Best Service Rate: {best_service['fleet_size']} vehicles ({best_service['service_rate']:.1f}%)")
        print(f"   🚗 Best Fleet Utilization: {best_utilization['fleet_size']} vehicles ({best_utilization['utilization_rate']:.1f}%)")
        print(f"   ⚡ Most Efficient: {most_efficient['fleet_size']} vehicles ({most_efficient['efficiency_score']:.2f}% per vehicle)")

        print(f"\n? DETAILED FLEET COMPARISON:")
        print("-" * 70)
        print(f"{'Fleet Size':<12} {'Service Rate':<12} {'Utilization':<12} {'Active Vehs':<12} {'Efficiency':<12}")
        print("-" * 70)

        for result in sorted(valid_results, key=lambda x: x['fleet_size']):
            print(f"{result['fleet_size']:<12} {result['service_rate']:<11.1f}% "
                  f"{result['utilization_rate']:<11.1f}% {result['active_vehicles']:<12} "
                  f"{result['efficiency_score']:<11.2f}")

        print(f"\n🎯 STRATEGIC RECOMMENDATIONS:")
        print("-" * 50)

        # Analyze each fleet size
        for result in sorted(valid_results, key=lambda x: x['fleet_size']):
            fleet_size = result['fleet_size']
            service_rate = result['service_rate']
            utilization = result['utilization_rate']

            print(f"\n🚌 {fleet_size} Vehicles Fleet:")

            # Service quality assessment
            if service_rate >= 80:
                service_quality = "Excellent service quality"
            elif service_rate >= 60:
                service_quality = "Good service quality"
            elif service_rate >= 40:
                service_quality = "Moderate service quality"
            else:
                service_quality = "Poor service quality"

            # Utilization assessment
            if utilization >= 75:
                util_assessment = "High utilization - fleet working efficiently"
            elif utilization >= 50:
                util_assessment = "Moderate utilization - balanced workload"
            elif utilization >= 25:
                util_assessment = "Low utilization - fleet may be oversized"
            else:
                util_assessment = "Very low utilization - significant overcapacity"

            print(f"   📈 Service: {service_quality} ({service_rate:.1f}%)")
            print(f"   🔧 Fleet: {util_assessment} ({utilization:.1f}%)")

            # Specific recommendations
            recommendations = []

            if service_rate >= 75 and utilization >= 60:
                recommendations.append("✅ Optimal configuration - good balance")
            elif service_rate >= 75 and utilization < 40:
                recommendations.append("⚠️ Oversized fleet - consider reducing")
            elif service_rate < 50 and utilization >= 75:
                recommendations.append("⚠️ Undersized fleet - consider increasing")
            elif service_rate < 30:
                recommendations.append("❌ Critical service level - major capacity increase needed")

            # Peak performance analysis
            morning_peak = result['morning_service_rate']
            afternoon_peak = result['afternoon_service_rate']

            if morning_peak >= 70:
                recommendations.append("🌅 Handles morning peak well")
            elif morning_peak >= 40:
                recommendations.append("🌅 Moderate morning peak performance")
            else:
                recommendations.append("🌅 Poor morning peak performance")

            if afternoon_peak >= 70:
                recommendations.append("🌆 Handles afternoon peak well")
            elif afternoon_peak >= 40:
                recommendations.append("🌆 Moderate afternoon peak performance")
            else:
                recommendations.append("🌆 Poor afternoon peak performance")

            for rec in recommendations:
                print(f"   💡 {rec}")

        print(f"\n🏛️ STRATEGIC IMPLEMENTATION PLAN:")
        print("-" * 50)

        # Determine best strategy based on results
        if len(valid_results) >= 2:
            sorted_by_service = sorted(valid_results, key=lambda x: x['service_rate'], reverse=True)

            if sorted_by_service[0]['service_rate'] > 70:
                recommended_fleet = sorted_by_service[0]['fleet_size']
                print(f"🎯 RECOMMENDED FLEET SIZE: {recommended_fleet} vehicles")
                print(f"   Provides {sorted_by_service[0]['service_rate']:.1f}% service rate")
                print(f"   Achieves {sorted_by_service[0]['utilization_rate']:.1f}% fleet utilization")
            else:
                print(f"⚠️ ALL TESTED FLEET SIZES SHOW SUBOPTIMAL PERFORMANCE")
                print(f"   Consider testing larger fleet sizes or reviewing:")
                print(f"   • Demand patterns and timing")
                print(f"   • Vehicle routing algorithms")
                print(f"   • Service area coverage")
                print(f"   • Fleet dispatch strategies")

        print(f"\n📋 OPERATIONAL INSIGHTS:")
        print("-" * 50)

        # Calculate overall demand patterns
        total_requests = sum(r['total_requests'] for r in valid_results) / len(valid_results)
        total_morning = sum(r['morning_peak_requests'] for r in valid_results) / len(valid_results)
        total_afternoon = sum(r['afternoon_peak_requests'] for r in valid_results) / len(valid_results)

        morning_pct = (total_morning / total_requests * 100) if total_requests > 0 else 0
        afternoon_pct = (total_afternoon / total_requests * 100) if total_requests > 0 else 0

        print(f"• Average daily demand: {total_requests:.0f} requests")
        print(f"• Morning peak: {total_morning:.0f} requests ({morning_pct:.1f}% of daily demand)")
        print(f"• Afternoon peak: {total_afternoon:.0f} requests ({afternoon_pct:.1f}% of daily demand)")

        # Vehicle utilization insights
        avg_utilization = sum(r['utilization_rate'] for r in valid_results) / len(valid_results)
        print(f"• Average fleet utilization: {avg_utilization:.1f}%")

        if avg_utilization < 50:
            print(f"💡 Low utilization suggests fleet sizing or routing optimization opportunities")
        elif avg_utilization > 80:
            print(f"💡 High utilization indicates efficient fleet management")

        print(f"\n✅ FLEETPY ANALYSIS COMPLETE")

        # Save recommendations
        recommendations_data = {
            'analysis_date': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M'),
            'simulation_type': 'FleetPy Professional Simulation',
            'scenarios_analyzed': len(valid_results),
            'best_service_fleet_size': best_service['fleet_size'],
            'best_service_rate': best_service['service_rate'],
            'most_efficient_fleet_size': most_efficient['fleet_size'],
            'efficiency_score': most_efficient['efficiency_score'],
            'average_utilization': avg_utilization,
            'total_daily_requests': total_requests
        }

        recommendations_df = pd.DataFrame([recommendations_data])
        recommendations_df.to_csv(os.path.join(analysis_results_dir, "fleetpy_recommendations.csv"), index=False)
        print(f"📋 Recommendations saved to: {os.path.join(analysis_results_dir, 'fleetpy_recommendations.csv')}")

    else:
        print("❌ No valid simulation results found")
        print("💡 This may indicate:")
        print("   • FleetPy simulation configuration issues")
        print("   • No demand data or routing problems")
        print("   • Incorrect fleet or network setup")
        print(f"\n🔧 TROUBLESHOOTING STEPS:")
        print("   1. Check FleetPy configuration files")
        print("   2. Verify demand data exists and is properly formatted")
        print("   3. Ensure network routing is working")
        print("   4. Re-run simulation: python run_klagenfurt.py")

else:
    print("❌ No FleetPy results loaded")
    print("💡 To run the analysis:")
    print("   1. Execute: python run_klagenfurt.py")
    print("   2. Re-run this notebook")

print("=" * 80)